# Causal Discovery and Causal Feature Selection for Robust Prediction

This tutorial walks through the key ideas and algorithms at the intersection of causal inference and robust machine learning.
The tutorial is structured as follows:

1. **Pairwise causal discovery** — can we tell which variable causes which, from observational data alone?
2. **Multivariate causal discovery** — recovering the full causal graph over many variables
3. **Multivariate causal feature selection** — finding the Markov Blanket of a target variable
4. **CFS for robust prediction** — why causal parents are more stable predictors than correlated features
5. **Invariant Causal Predictors** — a principled method to find invariant features across environments

In [ ]:
import os
if not os.path.exists('zh03-causal-discovery-robust-predictions'):
    !git clone https://github.com/WinterSchool2026/zh03-causal-discovery-robust-predictions.git
import sys
sys.path.insert(0, 'zh03-causal-discovery-robust-predictions')

In [ ]:
pip install causal-learn

In [ ]:
# Standard library
import time

# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# causal-learn
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.GES import ges
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.cit import fisherz

# LinGAM (with automatic backend detection)
from src.linGAM import direct_lingam

# Data generation
from src.generate_scm import *

np.random.seed(2)


# ── Helpers: extract adjacency matrices from causal-learn ─────────────────
def get_adjacency_pc(cg, col_names):
    g = cg.G.graph
    n = len(col_names)
    adj = pd.DataFrame(0, index=col_names, columns=col_names)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if g[i, j] == -1 and g[j, i] == 1:
                adj.iloc[i, j] = 1
            elif g[i, j] == -1 and g[j, i] == -1:
                adj.iloc[i, j] = 1
    return adj


def get_adjacency_ges(record, col_names):
    g = record['G'].graph
    n = len(col_names)
    adj = pd.DataFrame(0, index=col_names, columns=col_names)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if g[i, j] == -1 and g[j, i] == 1:
                adj.iloc[i, j] = 1
            elif g[i, j] == -1 and g[j, i] == -1:
                adj.iloc[i, j] = 1
    return adj


def get_adjacency_lingam(B, col_names, threshold=0.05):
    """
    Convert a LinGAM B matrix to a binary directed adjacency DataFrame.
    B[i, j] != 0  =>  j causes i  =>  we set adj[j, i] = 1  (same as PC/GES convention).
    """
    n = len(col_names)
    adj = pd.DataFrame(0, index=col_names, columns=col_names)
    for i in range(n):
        for j in range(n):
            if i != j and abs(B[i, j]) > threshold:
                adj.iloc[j, i] = 1   # j → i
    return adj


---
## Section 2 — Multivariate Causal Discovery

Three main families of algorithms:

**Constraint-based** (PC, FCI): Start from a complete graph. Remove edges where conditional independence holds. Orient remaining edges using consistency rules (v-structures, acyclicity).  
**Score-based** (GES): Define a score over graphs (e.g., BIC). Search over DAG space to maximize it.  
**Functional** (LiNGAM, RESIT): Use properties of the noise distribution to orient edges beyond what independence tests allow.

### 2.2 Assumptions

All algorithms require some form of faithfulness: every conditional independence in the data is reflected as a d-separation in the graph, and vice versa. Most also assume **causal sufficiency** (no hidden common causes), except FCI which relaxes this.

### 2.3 Identifiability

In general, observational data can only identify the graph up to a **Markov Equivalence Class (MEC)** — a set of DAGs that encode the same conditional independencies. Constraint-based methods return a CPDAG (Completed Partially Directed DAG) representing the MEC. 

In [ ]:
# Create an SCM with 10 variables + target Y
d = 10
scm = SCMGenerator(d=d)
scm.fit(n_parents=1, n_childs=1, n_spouses=1, sparsity=0.2, noise_type='uniform')

# Visualize the true graph
plot_graphs_from_adj([scm.A], Y_idx_list=[d], plot_titles=['True causal graph'])

print(f"Number of variables: {d + 1} (X0...X{d-1} + Y)")
print(f"True parents of Y: {scm.parents_idx}")
print(f"True children of Y: {scm.children_idx}")
print(f"True spouses of Y: {scm.spouses_idx}")


In [ ]:
# Sample from the SCM
data = scm.sample(n_samples=10000)
data.head()


### 2.6 Step 2 — Run PC and GES and compare to ground truth

We run both algorithms on the full dataset (n=10,000) at α=0.05.

In [ ]:
X_full = data.values

# PC
start = time.time()
cg_pc = pc(X_full, alpha=0.05, indep_test=fisherz)
A_pc = get_adjacency_pc(cg_pc, data.columns)
time_pc = time.time() - start
print(f'PC      : {time_pc:.2f}s')

# GES (BIC)
start = time.time()
record_ges = ges(X_full)
A_ges = get_adjacency_ges(record_ges, data.columns)
time_ges = time.time() - start
print(f'GES     : {time_ges:.2f}s')

# LinGAM  — note: direct_lingam takes a DataFrame and a target name;
# here we want the full graph, so we pass the full data and use the last column as target
# The B matrix gives us the full graph directly.
start = time.time()
_, B_lingam, _ = direct_lingam(data, 'Y', threshold=0.05)
A_lingam = get_adjacency_lingam(B_lingam, data.columns)
time_lingam = time.time() - start
print(f'LinGAM  : {time_lingam:.2f}s')

# Plot all three side by side
plot_graphs_from_adj(
    [scm.A, A_pc.values, A_ges.values, A_lingam.values],
    Y_idx_list=[d, d, d, d],
    plot_titles=['Ground Truth',
                 f'PC  ({time_pc:.1f}s)',
                 f'GES ({time_ges:.1f}s)',
                 f'LinGAM ({time_lingam:.1f}s)']
)


In [ ]:
def graph_metrics(A_true, A_pred, label=''):
    """
    Compute skeleton-level precision, recall, F1 and SHD.
    We symmetrise both matrices (treat directed and undirected edges alike).
    """
    true  = ((A_true  + A_true.T)  > 0).astype(int)
    pred  = ((A_pred  + A_pred.T)  > 0).astype(int)
    np.fill_diagonal(true, 0); np.fill_diagonal(pred, 0)

    mask = np.triu(np.ones_like(true, dtype=bool), k=1)
    t, p = true[mask], pred[mask]

    tp = int(((p == 1) & (t == 1)).sum())
    fp = int(((p == 1) & (t == 0)).sum())
    fn = int(((p == 0) & (t == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    shd       = fp + fn
    return {'precision': precision, 'recall': recall, 'f1': f1, 'shd': shd}


print('Skeleton metrics at n=10 000')
print(f'{"Algorithm":<12}  {"Precision":>9}  {"Recall":>6}  {"F1":>6}  {"SHD":>5}')
print('-' * 48)
for label, A_pred in [('PC', A_pc.values), ('GES', A_ges.values), ('LinGAM', A_lingam.values)]:
    m = graph_metrics(scm.A, A_pred)
    print(f'{label:<12}  {m["precision"]:>9.3f}  {m["recall"]:>6.3f}  {m["f1"]:>6.3f}  {m["shd"]:>5}')


### 2.7 Step 3 — Effect of sample size on PC vs GES vs LinGAM

All three algorithms degrade with fewer samples, but for different reasons:
- **PC** degrades because CI tests lose power — edges are incorrectly kept or removed.
- **GES** degrades because the BIC score becomes a noisier estimate — local greedy moves follow noise.
- **LinGAM** degrades because negentropy estimates require enough data to distinguish cause from effect. With very few samples the causal order can flip.

LinGAM has an additional requirement: **non-Gaussian noise**. Our SCM uses uniform noise, so LinGAM's identifiability assumptions hold here. With Gaussian noise it would reduce to a CPDAG (same output as PC/GES).

We run all three at several sample sizes on the **same SCM** and compare F1 and SHD.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

d_values = [5, 10, 15]
sample_sizes = [100, 300, 500]
n_reps = 2
results_list = []

for d in d_values:
    print(f'Processing d={d}...')
    col_names_sweep = [f'V{i}' for i in range(d)] + ['Y']
    for n in sample_sizes:
        for rep in range(n_reps):
            np.random.seed(rep * 17 + d)

            scm_s = SCMGenerator(d=d)
            if rep%2==0:
                noise = 'uniform'
            else:
                noise = 'gaussian'
            scm_s.fit(n_parents=1, n_childs=1, n_spouses=1,
                      sparsity=0.2, noise_type=noise)
            df_s = scm_s.sample(n_samples=n)
            X_s  = df_s.values

            # PC
            cg_s   = pc(X_s, alpha=0.05, indep_test=fisherz)
            A_pc_s = get_adjacency_pc(cg_s, col_names_sweep).values
            m_pc_s = graph_metrics(scm_s.A, A_pc_s)

            # GES
            rec_s   = ges(X_s)
            A_ges_s = get_adjacency_ges(rec_s, col_names_sweep).values
            m_ges_s = graph_metrics(scm_s.A, A_ges_s)

            # LinGAM
            _, B_s, _ = direct_lingam(df_s, 'Y', threshold=0.05)
            A_lnm_s   = get_adjacency_lingam(B_s, col_names_sweep).values
            m_lnm_s   = graph_metrics(scm_s.A, A_lnm_s)

            results_list.append({
                'd': d, 'n': n,
                'pc_f1':  m_pc_s['f1'],  'pc_shd':  m_pc_s['shd'],
                'ges_f1': m_ges_s['f1'], 'ges_shd': m_ges_s['shd'],
                'lnm_f1': m_lnm_s['f1'], 'lnm_shd': m_lnm_s['shd'],
            })

df_results = pd.DataFrame(results_list)
summary = df_results.groupby(['d', 'n']).mean().reset_index()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, metric in enumerate(['f1', 'shd']):
    for j, (algo, label) in enumerate([('pc', 'PC'), ('ges', 'GES'), ('lnm', 'LinGAM')]):
        col   = f'{algo}_{metric}'
        pivot = summary.pivot(index='d', columns='n', values=col)
        cmap  = 'YlGnBu' if metric == 'f1' else 'YlOrRd'
        fmt   = '.2f'    if metric == 'f1' else '.1f'
        sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap, ax=axes[i, j])
        axes[i, j].set_title(f'{label} — Mean {metric.upper()}')

plt.suptitle('Skeleton recovery: PC vs GES vs LinGAM\n'
             '(rows = metric, columns = algorithm)', fontsize=13)
plt.tight_layout()
plt.show()
